# 03 _ Feature Engineering and Baseline Forecasting

**Project:** Corporacion Favorita Store Sales Forecasting  
**Project type:** Data Science / MLOps Portfolio Project  
**Notebook:** `notebooks/03_feature_engineering_baselines.ipynb`

---

## 1.Purpose

This notebook connects the business exploratory analysis with the first modeling stage.

The goal is to create a reproducible modeling dataset and establish simple forecasting baselines that advanced models must beat.


# 2.Notebook Objective

The objective of this notebook is to generate model-ready features, build leakage-safe baselines, and evaluate them over the same 16-day validation horizon used later by the machine learning models.

The notebook will:

- Load the processed base tables created in previous notebooks.
- Validate the prediction grain `date + store_nbr + family`.
- Create a temporal train/validation split.
- Build basic calendar and historical sales features.
- Prevent leakage in lag and rolling features.
- Evaluate simple baseline forecasts.
- Save reusable datasets, predictions, metrics, and baseline artifacts.


# 3.Scope.

## Scope.

This notebook defines the first reproducible modeling layer for the project.

It includes:

- Processed input validation.
- Temporal validation split.
- Calendar features.
- Historical lag and rolling features.
- Baseline model evaluation.
- Baseline error analysis.
- Saved feature tables and baseline artifacts.

It does not include:

- LightGBM training.
- Prophet training.
- Hyperparameter optimization.
- MLflow experiment tracking.
- API development.
- Deep exploratory analysis.


# 4.Imports and Path Setup

This section imports the required libraries, detects the project root, defines modeling paths, and validates that previous notebooks generated the expected inputs.

This notebook reads from `data/processed/` and writes reusable features, predictions, reports, and baseline model metadata.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.4f}".format)

SEED = 42
np.random.seed(SEED)

# Project root

def find_project_root(start_path=None, required_dirs=("data", "notebooks", "reports")):
    """Find the project root by walking upward from the current directory."""
    current_path = Path.cwd() if start_path is None else Path(start_path)
    current_path = current_path.resolve()

    for candidate_path in [current_path, *current_path.parents]:
        if all((candidate_path / directory).exists() for directory in required_dirs):
            return candidate_path

    raise FileNotFoundError(
        "Project root could not be detected. "
        f"Expected a parent directory containing: {required_dirs}. "
        f"Current working directory: {current_path}"
    )


PROJECT_ROOT = find_project_root()

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"
PREDICTIONS_DIR = DATA_DIR / "predictions"

# Report directories
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"

# Model directories
MODELS_DIR = PROJECT_ROOT / "models"
BASELINES_MODELS_DIR = MODELS_DIR / "baselines"

# Input files
TRAIN_BASE_PATH = PROCESSED_DIR / "train_base.parquet"
TEST_BASE_PATH = PROCESSED_DIR / "test_base.parquet"

# This notebook does not create folders.
required_dirs = {
    "DATA_DIR": DATA_DIR,
    "INTERIM_DIR": INTERIM_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "FEATURES_DIR": FEATURES_DIR,
    "PREDICTIONS_DIR": PREDICTIONS_DIR,
    "REPORTS_TABLES_DIR": REPORTS_TABLES_DIR,
    "REPORTS_FIGURES_DIR": REPORTS_FIGURES_DIR,
    "BASELINES_MODELS_DIR": BASELINES_MODELS_DIR,
}

missing_dirs = {name: path for name, path in required_dirs.items() if not path.exists()}

if missing_dirs:
    raise FileNotFoundError(
        "Missing required directories:\n"
        + "\n".join([f"- {name}: {path}" for name, path in missing_dirs.items()])
    )

required_files = {
    "TRAIN_BASE_PATH": TRAIN_BASE_PATH,
    "TEST_BASE_PATH": TEST_BASE_PATH,
}

missing_files = {
    name: path for name, path in required_files.items() if not path.exists()
}

if missing_files:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join([f"- {name}: {path}" for name, path in missing_files.items()])
    )

dependency_versions = pd.DataFrame(
    [
        {"package": "pandas", "version": pd.__version__},
        {"package": "numpy", "version": np.__version__},
    ]
)

display(dependency_versions)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Features directory: {FEATURES_DIR}")
print(f"Predictions directory: {PREDICTIONS_DIR}")
print(f"Reports tables directory: {REPORTS_TABLES_DIR}")
print(f"Reports figures directory: {REPORTS_FIGURES_DIR}")
print(f"Baseline models directory: {BASELINES_MODELS_DIR}")


# 5.Load Processed Modeling Inputs

This section loads the processed base tables created by previous notebooks and prepares them for feature engineering.


In [2]:
train_base = pd.read_parquet(TRAIN_BASE_PATH)
test_base = pd.read_parquet(TEST_BASE_PATH)

train_base["date"] = pd.to_datetime(train_base["date"])
test_base["date"] = pd.to_datetime(test_base["date"])

train_base = train_base.sort_values(["date", "store_nbr", "family"]).reset_index(
    drop=True
)
test_base = test_base.sort_values(["date", "store_nbr", "family"]).reset_index(
    drop=True
)

data_overview = pd.DataFrame(
    [
        {
            "dataset": "train_base",
            "rows": len(train_base),
            "columns": train_base.shape[1],
            "start_date": train_base["date"].min(),
            "end_date": train_base["date"].max(),
            "stores": train_base["store_nbr"].nunique(),
            "families": train_base["family"].nunique(),
            "has_target": "sales" in train_base.columns,
        },
        {
            "dataset": "test_base",
            "rows": len(test_base),
            "columns": test_base.shape[1],
            "start_date": test_base["date"].min(),
            "end_date": test_base["date"].max(),
            "stores": test_base["store_nbr"].nunique(),
            "families": test_base["family"].nunique(),
            "has_target": "sales" in test_base.columns,
        },
    ]
)

display(data_overview)
display(train_base.head())
display(test_base.head())


,dataset,rows,columns,start_date,end_date,stores,families,has_target
0,train_base,3000888,30,2013-01-01,2017-08-15,54,33,True
1,test_base,28512,29,2017-08-16,2017-08-31,54,33,False


,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,0,2013-01-01,1,AUTOMOTIVE,0.0000,0,Quito,Pichincha,D,13,93.1400,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
1,1,2013-01-01,1,BABY CARE,0.0000,0,Quito,Pichincha,D,13,93.1400,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
2,2,2013-01-01,1,BEAUTY,0.0000,0,Quito,Pichincha,D,13,93.1400,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
3,3,2013-01-01,1,BEVERAGES,0.0000,0,Quito,Pichincha,D,13,93.1400,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False
4,4,2013-01-01,1,BOOKS,0.0000,0,Quito,Pichincha,D,13,93.1400,True,True,True,1,0,1,1,0,0,0,1,0,0,0,1,0,True,True,False


,id,date,store_nbr,family,onpromotion,city,state,store_type,cluster,dcoilwtico,oil_row_exists_in_raw,dcoilwtico_missing_before_imputation,dcoilwtico_was_imputed,calendar_event_count,transferred_event_count,unique_locale_names,non_transferred_event_count,holiday_type_additional,holiday_type_bridge,holiday_type_event,holiday_type_holiday,holiday_type_transfer,holiday_type_work_day,holiday_locale_local,holiday_locale_national,holiday_locale_regional,is_calendar_event,has_non_transferred_event,has_transferred_event
0,3000888,2017-08-16,1,AUTOMOTIVE,0,Quito,Pichincha,D,13,46.8000,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
1,3000889,2017-08-16,1,BABY CARE,0,Quito,Pichincha,D,13,46.8000,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
2,3000890,2017-08-16,1,BEAUTY,2,Quito,Pichincha,D,13,46.8000,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
3,3000891,2017-08-16,1,BEVERAGES,20,Quito,Pichincha,D,13,46.8000,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False
4,3000892,2017-08-16,1,BOOKS,0,Quito,Pichincha,D,13,46.8000,True,False,False,0,0,0,0,0,0,0,0,0,0,0,0,0,False,False,False


# 6.Prediction Grain Validation

This section validates that the modeling data respects the expected `date + store_nbr + family` prediction grain.


In [3]:
PREDICTION_GRAIN = ["date", "store_nbr", "family"]
TARGET = "sales"
ID_COLUMN = "id"

required_train_columns = PREDICTION_GRAIN + [TARGET, "onpromotion"]
required_test_columns = PREDICTION_GRAIN + ["onpromotion"]

missing_train_columns = [
    col for col in required_train_columns if col not in train_base.columns
]
missing_test_columns = [
    col for col in required_test_columns if col not in test_base.columns
]

if missing_train_columns:
    raise ValueError(f"Missing required train columns: {missing_train_columns}")

if missing_test_columns:
    raise ValueError(f"Missing required test columns: {missing_test_columns}")

train_duplicates = train_base.duplicated(PREDICTION_GRAIN).sum()
test_duplicates = test_base.duplicated(PREDICTION_GRAIN).sum()

if train_duplicates > 0:
    raise ValueError(
        f"Duplicated prediction grain rows found in train_base: {train_duplicates}"
    )

if test_duplicates > 0:
    raise ValueError(
        f"Duplicated p[rediction grain rows found in test_base: {test_duplicates}"
    )

common_feature_columns = sorted(
    list(
        set(train_base.columns)
        .intersection(set(test_base.columns))
        .difference({ID_COLUMN})
    )
)

train_only_columns = sorted(
    list(set(train_base.columns).difference(set(test_base.columns)))
)
test_only_columns = sorted(
    list(set(test_base.columns).difference(set(train_base.columns)))
)

validation_summary = pd.DataFrame(
    [
        {
            "check": "prediction_grain",
            "result": "passed",
            "details": "date + store_nbr + family is unique in train and test",
        },
        {
            "check": "target_column",
            "result": (
                "passed"
                if TARGET in train_base.columns and TARGET not in test_base.columns
                else "review"
            ),
            "details": "sales is present in train_base and absent from test_base",
        },
        {
            "check": "train_duplicates",
            "result": "passed",
            "details": train_duplicates,
        },
        {
            "check": "test_duplicates",
            "result": "passed",
            "details": test_duplicates,
        },
        {
            "check": "common_columns_without_id",
            "result": "info",
            "details": len(common_feature_columns),
        },
        {
            "check": "train_only_columns",
            "result": "info",
            "details": train_only_columns,
        },
        {
            "check": "test_only_columns",
            "result": "info",
            "details": test_only_columns,
        },
    ]
)


display(validation_summary)


,check,result,details
0,prediction_grain,passed,date + store_nbr + family is unique in train a...
1,target_column,passed,sales is present in train_base and absent from...
2,train_duplicates,passed,0
3,test_duplicates,passed,0
4,common_columns_without_id,info,28
5,train_only_columns,info,[sales]
6,test_only_columns,info,[]


### Section conclusion

The processed modeling inputs are consistent with the expected forecasting grain.

Each row represents daily sales for one `store_nbr` and one `family`. The target variable `sales` is available only in the training data, while the Kaggle test set keeps the same prediction structure without the target.

No duplicated rows were found at the `date + store_nbr + family` level, so the dataset is ready for temporal splitting and baseline feature engineering.


# 7.Temporal Train Validation Split

This section creates the 16-day validation window used for baseline evaluation and later model comparison.


In [4]:
FORECAST_HORIZON_DAYS = 16

TRAIN_END_DATE = pd.Timestamp("2017-07-30")
VALID_START_DATE = pd.Timestamp("2017-07-31")
VALID_END_DATE = pd.Timestamp("2017-08-15")

TEST_START_DATE = test_base["date"].min()
TEST_END_DATE = test_base["date"].max()

train_model_base = train_base.loc[train_base["date"] <= TRAIN_END_DATE].copy()

valid_base = train_base.loc[
    (train_base["date"] >= VALID_START_DATE) & (train_base["date"] <= VALID_END_DATE)
].copy()

expected_daily_rows = train_base["store_nbr"].nunique() * train_base["family"].nunique()

expected_valid_rows = expected_daily_rows * FORECAST_HORIZON_DAYS

split_checks = {
    "train_model_has_rows": len(train_model_base) > 0,
    "valid_has_rows": len(valid_base) > 0,
    "train_before_validation": train_model_base["date"].max()
    < valid_base["date"].min(),
    "validation_horizon_is_16_days": valid_base["date"].nunique()
    == FORECAST_HORIZON_DAYS,
    "validation_rows_match_expected_grid": len(valid_base) == expected_valid_rows,
    "test_horizon_is_16_days": test_base["date"].nunique() == FORECAST_HORIZON_DAYS,
}

failed_checks = [
    check_name for check_name, check_passed in split_checks.items() if not check_passed
]

if failed_checks:
    raise ValueError(f"Temporal split checks failed: {failed_checks}")

temporal_split_summary = pd.DataFrame(
    [
        {
            "dataset": "train_model_base",
            "start_date": train_model_base["date"].min(),
            "end_date": train_model_base["date"].max(),
            "days": train_model_base["date"].nunique(),
            "rows": len(train_model_base),
            "stores": train_model_base["store_nbr"].nunique(),
            "families": train_model_base["family"].nunique(),
            "total_sales": train_model_base["sales"].sum(),
        },
        {
            "dataset": "valid_base",
            "start_date": valid_base["date"].min(),
            "end_date": valid_base["date"].max(),
            "days": valid_base["date"].nunique(),
            "rows": len(valid_base),
            "stores": valid_base["store_nbr"].nunique(),
            "families": valid_base["family"].nunique(),
            "total_sales": valid_base["sales"].sum(),
        },
        {
            "dataset": "test_base",
            "start_date": test_base["date"].min(),
            "end_date": test_base["date"].max(),
            "days": test_base["date"].nunique(),
            "rows": len(test_base),
            "stores": test_base["store_nbr"].nunique(),
            "families": test_base["family"].nunique(),
            "total_sales": np.nan,
        },
    ]
)

display(temporal_split_summary)

print("Temporal split created successfully.")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"Forecast origin for validation: {TRAIN_END_DATE.date()}")


,dataset,start_date,end_date,days,rows,stores,families,total_sales
0,train_model_base,2013-01-01,2017-07-30,1668,2972376,54,33,"1,060,325,772.4214"
1,valid_base,2017-07-31,2017-08-15,16,28512,54,33,"13,319,179.7816"
2,test_base,2017-08-16,2017-08-31,16,28512,54,33,NaN


Temporal split created successfully.
Forecast horizon: 16 days
Forecast origin for validation: 2017-07-30


### Section conclusion

The modeling split follows a strict time-based validation strategy.

The model training period ends on `2017-07-30`, and the validation period covers the following 16 days, from `2017-07-31` to `2017-08-15`. This mirrors the Kaggle test horizon from `2017-08-16` to `2017-08-31`.

This split avoids random sampling and reproduces the real forecasting setting: models can only learn from the past and must predict a future 16-day window.


# 8.Feature Policy

This section defines the columns that can be used safely by the baseline feature set.


In [5]:
KEY_COLUMNS = ["date", "store_nbr", "family"]
TARGET_COLUMN = "sales"
ID_COLUMN = "id"

BASELINE_EXCLUDED_COLUMNS = [
    ID_COLUMN,
    TARGET_COLUMN,
]

transaction_like_columns = [
    col for col in train_base.columns if "transaction" in col.lower()
]

common_columns = sorted(
    list(set(train_base.columns).intersection(set(test_base.columns)))
)

available_modeling_columns = [
    col for col in common_columns if col not in BASELINE_EXCLUDED_COLUMNS
]

known_future_columns = [
    col
    for col in available_modeling_columns
    if col
    in [
        "date",
        "store_nbr",
        "family",
        "onpromotion",
        "city",
        "state",
        "store_type",
        "cluster",
        "dcoilwtico",
        "oil_row_exists_in_raw",
        "dcoilwtico_missing_before_imputation",
        "dcoilwtico_was_imputed",
        "calendar_event_count",
        "transferred_event_count",
        "unique_locale_names",
        "non_transferred_event_count",
        "is_calendar_event",
        "has_non_transferred_event",
        "has_transferred_event",
    ]
    or col.startswith("holiday_type_")
    or col.startswith("holiday_locale_")
]

reserved_for_future_modeling = [
    col for col in known_future_columns if col not in KEY_COLUMNS
]

baseline_feature_policy = pd.DataFrame(
    [
        {
            "feature_group": "prediction_keys",
            "columns": KEY_COLUMNS,
            "used_in_baselines": True,
            "reason": "Defines the forecasting grain.",
        },
        {
            "feature_group": "target",
            "columns": [TARGET_COLUMN],
            "used_in_baselines": False,
            "reason": "Target variable. Only used for training history and evaluation.",
        },
        {
            "feature_group": "identifier",
            "columns": [ID_COLUMN] if ID_COLUMN in common_columns else [],
            "used_in_baselines": False,
            "reason": "Technical row identifier. Not predictive.",
        },
        {
            "feature_group": "known_future_features",
            "columns": reserved_for_future_modeling,
            "used_in_baselines": False,
            "reason": "Available in train and test. Preserved for future models such as LightGBM.",
        },
        {
            "feature_group": "transactions",
            "columns": transaction_like_columns,
            "used_in_baselines": False,
            "reason": "Excluded because future transactions are not available at prediction time.",
        },
        {
            "feature_group": "historical_sales_features",
            "columns": [
                "sales_lag_16",
                "sales_lag_21",
                "sales_lag_28",
                "sales_rolling_mean_7_shift_16",
                "sales_rolling_mean_14_shift_16",
                "sales_rolling_mean_28_shift_16",
            ],
            "used_in_baselines": True,
            "reason": "Created later using only sales available before the forecast horizon.",
        },
    ]
)

leakage_rules = pd.DataFrame(
    [
        {
            "rule_id": 1,
            "rule": "Validation must be strictly later than training.",
            "application": "The validation window starts after TRAIN_END_DATE.",
        },
        {
            "rule_id": 2,
            "rule": "Historical sales features must not use validation sales.",
            "application": f"Sales lags and rolling features must use at least a {FORECAST_HORIZON_DAYS}-day shift.",
        },
        {
            "rule_id": 3,
            "rule": "Baseline statistics must be learned only from train_model_base.",
            "application": "Validation rows can only receive statistics computed from the training period.",
        },
        {
            "rule_id": 4,
            "rule": "The target column must never be used as a direct feature.",
            "application": "sales is only used to build past-dependent features and evaluate predictions.",
        },
        {
            "rule_id": 5,
            "rule": "Transactions must not be used as predictive input.",
            "application": "transactions is historical-only and not available for the future Kaggle test period.",
        },
        {
            "rule_id": 6,
            "rule": "Test data can validate schema but not model performance.",
            "application": "The Kaggle test target is unknown and must not influence baseline selection.",
        },
    ]
)

display(baseline_feature_policy)
display(leakage_rules)

print(f"Common columns available in train and test: {len(common_columns)}")
print(f"Known future columns preserved for modeling: {len(known_future_columns)}")
print(f"Transaction-like columns detected: {transaction_like_columns}")


,feature_group,columns,used_in_baselines,reason
0,prediction_keys,"[date, store_nbr, family]",True,Defines the forecasting grain.
1,target,[sales],False,Target variable. Only used for training histor...
2,identifier,[id],False,Technical row identifier. Not predictive.
3,known_future_features,"[calendar_event_count, city, cluster, dcoilwti...",False,Available in train and test. Preserved for fut...
4,transactions,[],False,Excluded because future transactions are not a...
5,historical_sales_features,"[sales_lag_16, sales_lag_21, sales_lag_28, sal...",True,Created later using only sales available befor...


,rule_id,rule,application
0,1,Validation must be strictly later than training.,The validation window starts after TRAIN_END_D...
1,2,Historical sales features must not use validat...,Sales lags and rolling features must use at le...
2,3,Baseline statistics must be learned only from ...,Validation rows can only receive statistics co...
3,4,The target column must never be used as a dire...,sales is only used to build past-dependent fea...
4,5,Transactions must not be used as predictive in...,transactions is historical-only and not availa...
5,6,Test data can validate schema but not model pe...,The Kaggle test target is unknown and must not...


Common columns available in train and test: 29
Known future columns preserved for modeling: 28
Transaction-like columns detected: []


### Section conclusion

The notebook separates prediction keys, target variables, known future features and historical sales features.

For baseline forecasting, the most important design rule is to avoid using validation sales when creating lags, rolling features or historical averages. Any target-derived feature must be computed using only information available before the forecast horizon.

`transactions` is excluded from predictive modeling because it is not available for the future test period.


# 9.Calendar Feature Engineering

This section creates reusable calendar features available for both historical training rows and future inference rows.


In [ ]:
def add_calendar_features(df):
    df = df.copy()

    df["day_of_week"] = df["date"].dt.dayofweek.astype("int16")
    df["day_of_month"] = df["date"].dt.day.astype("int16")
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month.astype("int16")
    df["year"] = df["date"].dt.year.astype("int16")
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype("int8")
    df["is_month_start"] = df["date"].dt.is_month_start.astype("int8")
    df["is_month_end"] = df["date"].dt.is_month_end.astype("int8")

    return df


CALENDAR_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

train_model_fe = add_calendar_features(train_model_base)
valid_fe = add_calendar_features(valid_base)
test_fe = add_calendar_features(test_base)
train_full_fe = add_calendar_features(train_base)

calendar_feature_checks = pd.DataFrame(
    [
        {
            "dataset": "train_model_fe",
            "rows": len(train_model_fe),
            "start_date": train_model_fe["date"].min(),
            "end_date": train_model_fe["date"].max(),
            "missing_calendar_values": train_model_fe[CALENDAR_FEATURES]
            .isna()
            .sum()
            .sum(),
        },
        {
            "dataset": "valid_fe",
            "rows": len(valid_fe),
            "start_date": valid_fe["date"].min(),
            "end_date": valid_fe["date"].max(),
            "missing_calendar_values": valid_fe[CALENDAR_FEATURES].isna().sum().sum(),
        },
        {
            "dataset": "test_fe",
            "rows": len(test_fe),
            "start_date": test_fe["date"].min(),
            "end_date": test_fe["date"].max(),
            "missing_calendar_values": test_fe[CALENDAR_FEATURES].isna().sum().sum(),
        },
        {
            "dataset": "train_full_fe",
            "rows": len(train_full_fe),
            "start_date": train_full_fe["date"].min(),
            "end_date": train_full_fe["date"].max(),
            "missing_calendar_values": train_full_fe[CALENDAR_FEATURES]
            .isna()
            .sum()
            .sum(),
        },
    ]
)

display(calendar_feature_checks)
display(train_model_fe[["date"] + CALENDAR_FEATURES].head())
display(valid_fe[["date"] + CALENDAR_FEATURES].head())
display(test_fe[["date"] + CALENDAR_FEATURES].head())


,dataset,rows,start_date,end_date,missing_calendar_values
0,train_model_fe,2972376,2013-01-01,2017-07-30,0
1,valid_fe,28512,2017-07-31,2017-08-15,0
2,test_fe,28512,2017-08-16,2017-08-31,0
3,train_full_fe,3000888,2013-01-01,2017-08-15,0


,date,day_of_week,day_of_month,week_of_year,month,year,is_weekend,is_month_start,is_month_end
0,2013-01-01,1,1,1,1,2013,0,1,0
1,2013-01-01,1,1,1,1,2013,0,1,0
2,2013-01-01,1,1,1,1,2013,0,1,0
3,2013-01-01,1,1,1,1,2013,0,1,0
4,2013-01-01,1,1,1,1,2013,0,1,0


,date,day_of_week,day_of_month,week_of_year,month,year,is_weekend,is_month_start,is_month_end
2972376,2017-07-31,0,31,31,7,2017,0,0,1
2972377,2017-07-31,0,31,31,7,2017,0,0,1
2972378,2017-07-31,0,31,31,7,2017,0,0,1
2972379,2017-07-31,0,31,31,7,2017,0,0,1
2972380,2017-07-31,0,31,31,7,2017,0,0,1


,date,day_of_week,day_of_month,week_of_year,month,year,is_weekend,is_month_start,is_month_end
0,2017-08-16,2,16,33,8,2017,0,0,0
1,2017-08-16,2,16,33,8,2017,0,0,0
2,2017-08-16,2,16,33,8,2017,0,0,0
3,2017-08-16,2,16,33,8,2017,0,0,0
4,2017-08-16,2,16,33,8,2017,0,0,0


### Section conclusion

Basic calendar features were created from the `date` column for training, validation, full training and test datasets.

These features are safe for forecasting because future dates are known in advance. They allow future models to learn weekly, monthly and seasonal demand patterns without using any future sales information.


# 10.Historical Sales Feature Engineering

This section creates leakage-safe lag and rolling sales features using only information available before the forecast horizon.


In [ ]:
GROUP_COLUMNS = ["store_nbr", "family"]

LAG_DAYS = [16, 21, 28]
ROLLING_WINDOWS = [7, 14, 28]
ROLLING_SHIFT_DAYS = FORECAST_HORIZON_DAYS

LAG_FEATURES = [f"sales_lag_{lag}" for lag in LAG_DAYS]

ROLLING_FEATURES = [
    f"sales_rolling_mean_{window}_shift_{ROLLING_SHIFT_DAYS}"
    for window in ROLLING_WINDOWS
]

HISTORICAL_SALES_FEATURES = LAG_FEATURES + ROLLING_FEATURES


def build_lag_lookup(history_df, lag_days):
    lookups = {}

    history = history_df[KEY_COLUMNS + [TARGET_COLUMN]].copy()
    history = history.dropna(subset=[TARGET_COLUMN])

    for lag in lag_days:
        feature_name = f"sales_lag_{lag}"

        lookup = history.copy()
        lookup["date"] = lookup["date"] + pd.Timedelta(days=lag)
        lookup = lookup.rename(columns={TARGET_COLUMN: feature_name})
        lookup = lookup[KEY_COLUMNS + [feature_name]]

        duplicate_rows = lookup.duplicated(KEY_COLUMNS).sum()

        if duplicate_rows > 0:
            raise ValueError(
                f"Duplicated rows found in lookup for {feature_name}: {duplicate_rows}"
            )

        lookups[feature_name] = lookup

    return lookups


def build_rolling_lookup(history_df, rolling_windows, shift_days):
    lookups = {}

    history = history_df[KEY_COLUMNS + [TARGET_COLUMN]].copy()
    history = history.dropna(subset=[TARGET_COLUMN])
    history = history.sort_values(GROUP_COLUMNS + ["date"]).reset_index(drop=True)

    for window in rolling_windows:
        feature_name = f"sales_rolling_mean_{window}_shift_{shift_days}"

        rolling_parts = []

        for _, group in history.groupby(GROUP_COLUMNS, sort=False):
            group = group.sort_values("date").copy()

            rolling_values = (
                group.set_index("date")[TARGET_COLUMN]
                .rolling(f"{window}D", min_periods=1)
                .mean()
                .to_numpy()
            )

            group_lookup = group[KEY_COLUMNS].copy()
            group_lookup[feature_name] = rolling_values

            rolling_parts.append(group_lookup)

        lookup = pd.concat(rolling_parts, ignore_index=True)
        lookup["date"] = lookup["date"] + pd.Timedelta(days=shift_days)
        lookup = lookup[KEY_COLUMNS + [feature_name]]

        duplicate_rows = lookup.duplicated(KEY_COLUMNS).sum()

        if duplicate_rows > 0:
            raise ValueError(
                f"Duplicated rows found in lookup for {feature_name}: {duplicate_rows}"
            )

        lookups[feature_name] = lookup

    return lookups


def apply_feature_lookups(target_df, feature_lookups):
    output_df = target_df.copy()

    for feature_name, lookup in feature_lookups.items():
        output_df = output_df.merge(
            lookup,
            on=KEY_COLUMNS,
            how="left",
        )

        output_df[feature_name] = output_df[feature_name].astype("float32")

    return output_df


train_origin_lag_lookups = build_lag_lookup(
    history_df=train_model_fe,
    lag_days=LAG_DAYS,
)

train_origin_rolling_lookups = build_rolling_lookup(
    history_df=train_model_fe,
    rolling_windows=ROLLING_WINDOWS,
    shift_days=ROLLING_SHIFT_DAYS,
)

train_origin_feature_lookups = {
    **train_origin_lag_lookups,
    **train_origin_rolling_lookups,
}

full_origin_lag_lookups = build_lag_lookup(
    history_df=train_full_fe,
    lag_days=LAG_DAYS,
)

full_origin_rolling_lookups = build_rolling_lookup(
    history_df=train_full_fe,
    rolling_windows=ROLLING_WINDOWS,
    shift_days=ROLLING_SHIFT_DAYS,
)

full_origin_feature_lookups = {
    **full_origin_lag_lookups,
    **full_origin_rolling_lookups,
}

train_model_fe = apply_feature_lookups(
    target_df=train_model_fe,
    feature_lookups=train_origin_feature_lookups,
)

valid_fe = apply_feature_lookups(
    target_df=valid_fe,
    feature_lookups=train_origin_feature_lookups,
)

train_full_fe = apply_feature_lookups(
    target_df=train_full_fe,
    feature_lookups=full_origin_feature_lookups,
)

test_fe = apply_feature_lookups(
    target_df=test_fe,
    feature_lookups=full_origin_feature_lookups,
)

historical_feature_missing_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "dataset": dataset_name,
                "feature": HISTORICAL_SALES_FEATURES,
                "missing_rows": dataset[HISTORICAL_SALES_FEATURES].isna().sum().values,
                "missing_pct": (
                    dataset[HISTORICAL_SALES_FEATURES].isna().mean().values * 100
                ).round(4),
            }
        )
        for dataset_name, dataset in [
            ("train_model_fe", train_model_fe),
            ("valid_fe", valid_fe),
            ("train_full_fe", train_full_fe),
            ("test_fe", test_fe),
        ]
    ],
    ignore_index=True,
)

source_safety_checks = []

for lag in LAG_DAYS:
    source_safety_checks.append(
        {
            "feature": f"sales_lag_{lag}",
            "validation_latest_source_date": VALID_END_DATE - pd.Timedelta(days=lag),
            "validation_safe": VALID_END_DATE - pd.Timedelta(days=lag)
            <= TRAIN_END_DATE,
            "test_latest_source_date": TEST_END_DATE - pd.Timedelta(days=lag),
            "test_safe": TEST_END_DATE - pd.Timedelta(days=lag)
            <= train_base["date"].max(),
        }
    )

for window in ROLLING_WINDOWS:
    source_safety_checks.append(
        {
            "feature": f"sales_rolling_mean_{window}_shift_{ROLLING_SHIFT_DAYS}",
            "validation_latest_source_date": VALID_END_DATE
            - pd.Timedelta(days=ROLLING_SHIFT_DAYS),
            "validation_safe": VALID_END_DATE - pd.Timedelta(days=ROLLING_SHIFT_DAYS)
            <= TRAIN_END_DATE,
            "test_latest_source_date": TEST_END_DATE
            - pd.Timedelta(days=ROLLING_SHIFT_DAYS),
            "test_safe": TEST_END_DATE - pd.Timedelta(days=ROLLING_SHIFT_DAYS)
            <= train_base["date"].max(),
        }
    )

source_safety_checks = pd.DataFrame(source_safety_checks)

display(historical_feature_missing_summary)
display(source_safety_checks)
display(valid_fe[KEY_COLUMNS + [TARGET_COLUMN] + HISTORICAL_SALES_FEATURES].head())
display(test_fe[KEY_COLUMNS + HISTORICAL_SALES_FEATURES].head())


,dataset,feature,missing_rows,missing_pct
0,train_model_fe,sales_lag_16,35640,1.1990
1,train_model_fe,sales_lag_21,44550,1.4988
2,train_model_fe,sales_lag_28,57024,1.9185
3,train_model_fe,sales_rolling_mean_7_shift_16,35640,1.1990
4,train_model_fe,sales_rolling_mean_14_shift_16,35640,1.1990
5,train_model_fe,sales_rolling_mean_28_shift_16,35640,1.1990
6,valid_fe,sales_lag_16,0,0.0000
7,valid_fe,sales_lag_21,0,0.0000
8,valid_fe,sales_lag_28,0,0.0000
9,valid_fe,sales_rolling_mean_7_shift_16,0,0.0000


,feature,validation_latest_source_date,validation_safe,test_latest_source_date,test_safe
0,sales_lag_16,2017-07-30,True,2017-08-15,True
1,sales_lag_21,2017-07-25,True,2017-08-10,True
2,sales_lag_28,2017-07-18,True,2017-08-03,True
3,sales_rolling_mean_7_shift_16,2017-07-30,True,2017-08-15,True
4,sales_rolling_mean_14_shift_16,2017-07-30,True,2017-08-15,True
5,sales_rolling_mean_28_shift_16,2017-07-30,True,2017-08-15,True


,date,store_nbr,family,sales,sales_lag_16,sales_lag_21,sales_lag_28,sales_rolling_mean_7_shift_16,sales_rolling_mean_14_shift_16,sales_rolling_mean_28_shift_16
0,2017-07-31,1,AUTOMOTIVE,8.0000,6.0000,3.0000,0.0000,5.4286,4.5000,4.5357
1,2017-07-31,1,BABY CARE,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,2017-07-31,1,BEAUTY,3.0000,5.0000,3.0000,5.0000,4.1429,3.7143,4.0714
3,2017-07-31,1,BEVERAGES,"2,414.0000","2,183.0000","2,338.0000","2,526.0000","2,168.2856","2,215.9285","2,207.4644"
4,2017-07-31,1,BOOKS,1.0000,0.0000,0.0000,1.0000,0.4286,0.3571,0.2143


,date,store_nbr,family,sales_lag_16,sales_lag_21,sales_lag_28,sales_rolling_mean_7_shift_16,sales_rolling_mean_14_shift_16,sales_rolling_mean_28_shift_16
0,2017-08-16,1,AUTOMOTIVE,8.0000,2.0000,7.0000,5.2857,5.2143,4.8571
1,2017-08-16,1,BABY CARE,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,2017-08-16,1,BEAUTY,3.0000,3.0000,3.0000,3.0000,2.6429,3.1786
3,2017-08-16,1,BEVERAGES,"2,414.0000","2,242.0000","2,369.0000","2,118.1428","2,149.2856","2,175.4285"
4,2017-08-16,1,BOOKS,1.0000,0.0000,0.0000,0.1429,0.0714,0.2143


### Section conclusion

Historical sales features were created using calendar-based lookups instead of row-position shifts.

For validation, all lag and rolling features are computed from `train_model_fe`, which ends on `2017-07-30`. This prevents validation sales from leaking into the feature set.

For the Kaggle test period, historical features are computed from the full available training history ending on `2017-08-15`, which matches the real forecasting setup.

The resulting lag and rolling features are time-safe and can be reused as a foundation for future LightGBM modeling.


# 11.Baseline Forecast Generation

This section generates simple baseline forecasts for the validation window.


In [ ]:
baseline_predictions_valid = valid_fe[
    KEY_COLUMNS + [TARGET_COLUMN, "onpromotion", "day_of_week"]
].copy()

baseline_predictions_valid["horizon_day"] = (
    baseline_predictions_valid["date"] - VALID_START_DATE
).dt.days + 1

baseline_predictions_valid["baseline_zero"] = 0.0


last_observed_lookup = train_model_fe.loc[
    train_model_fe["date"] == TRAIN_END_DATE,
    ["store_nbr", "family", TARGET_COLUMN],
].rename(columns={TARGET_COLUMN: "baseline_last_observed"})

baseline_predictions_valid = baseline_predictions_valid.merge(
    last_observed_lookup,
    on=["store_nbr", "family"],
    how="left",
)


last_week_pattern = train_model_fe.loc[
    (train_model_fe["date"] >= TRAIN_END_DATE - pd.Timedelta(days=6))
    & (train_model_fe["date"] <= TRAIN_END_DATE),
    ["store_nbr", "family", "day_of_week", TARGET_COLUMN],
].copy()

last_week_pattern = last_week_pattern.rename(
    columns={TARGET_COLUMN: "baseline_last_7_day_pattern"}
)

pattern_duplicates = last_week_pattern.duplicated(
    ["store_nbr", "family", "day_of_week"]
).sum()

if pattern_duplicates > 0:
    raise ValueError(
        f"Duplicated rows found in last week pattern lookup: {pattern_duplicates}"
    )

baseline_predictions_valid = baseline_predictions_valid.merge(
    last_week_pattern,
    on=["store_nbr", "family", "day_of_week"],
    how="left",
)


rolling_28_origin_lookup = (
    train_model_fe.loc[
        (train_model_fe["date"] >= TRAIN_END_DATE - pd.Timedelta(days=27))
        & (train_model_fe["date"] <= TRAIN_END_DATE)
    ]
    .groupby(["store_nbr", "family"], as_index=False)[TARGET_COLUMN]
    .mean()
    .rename(columns={TARGET_COLUMN: "baseline_rolling_mean_28_origin"})
)

baseline_predictions_valid = baseline_predictions_valid.merge(
    rolling_28_origin_lookup,
    on=["store_nbr", "family"],
    how="left",
)


weekday_average_lookup = (
    train_model_fe.groupby(["store_nbr", "family", "day_of_week"], as_index=False)[
        TARGET_COLUMN
    ]
    .mean()
    .rename(columns={TARGET_COLUMN: "baseline_store_family_weekday_avg"})
)

baseline_predictions_valid = baseline_predictions_valid.merge(
    weekday_average_lookup,
    on=["store_nbr", "family", "day_of_week"],
    how="left",
)


BASELINE_COLUMNS = [
    "baseline_zero",
    "baseline_last_observed",
    "baseline_last_7_day_pattern",
    "baseline_rolling_mean_28_origin",
    "baseline_store_family_weekday_avg",
]

for col in BASELINE_COLUMNS:
    baseline_predictions_valid[col] = baseline_predictions_valid[col].fillna(0)
    baseline_predictions_valid[col] = baseline_predictions_valid[col].clip(lower=0)
    baseline_predictions_valid[col] = baseline_predictions_valid[col].astype("float32")

baseline_missing_summary = pd.DataFrame(
    {
        "baseline": BASELINE_COLUMNS,
        "missing_rows": baseline_predictions_valid[BASELINE_COLUMNS]
        .isna()
        .sum()
        .values,
        "missing_pct": (
            baseline_predictions_valid[BASELINE_COLUMNS].isna().mean().values * 100
        ).round(4),
        "min_prediction": baseline_predictions_valid[BASELINE_COLUMNS].min().values,
        "max_prediction": baseline_predictions_valid[BASELINE_COLUMNS].max().values,
    }
)

baseline_preview = baseline_predictions_valid[
    KEY_COLUMNS + [TARGET_COLUMN, "horizon_day"] + BASELINE_COLUMNS
].head(10)

display(baseline_missing_summary)
display(baseline_preview)

print(f"Validation prediction rows: {len(baseline_predictions_valid):,}")
print(f"Baselines created: {len(BASELINE_COLUMNS)}")


,baseline,missing_rows,missing_pct,min_prediction,max_prediction
0,baseline_zero,0,0.0000,0.0000,0.0000
1,baseline_last_observed,0,0.0000,0.0000,"18,340.0000"
2,baseline_last_7_day_pattern,0,0.0000,0.0000,"18,340.0000"
3,baseline_rolling_mean_28_origin,0,0.0000,0.0000,"11,334.8926"
4,baseline_store_family_weekday_avg,0,0.0000,0.0000,"13,904.9346"


,date,store_nbr,family,sales,horizon_day,baseline_zero,baseline_last_observed,baseline_last_7_day_pattern,baseline_rolling_mean_28_origin,baseline_store_family_weekday_avg
0,2017-07-31,1,AUTOMOTIVE,8.0000,1,0.0000,1.0000,4.0000,4.5714,3.2773
1,2017-07-31,1,BABY CARE,0.0000,1,0.0000,0.0000,0.0000,0.0000,0.0000
2,2017-07-31,1,BEAUTY,3.0000,1,0.0000,2.0000,1.0000,3.2500,2.6345
3,2017-07-31,1,BEVERAGES,"2,414.0000",1,0.0000,"1,212.0000","2,158.0000","2,179.4285","1,693.5714"
4,2017-07-31,1,BOOKS,1.0000,1,0.0000,0.0000,0.0000,0.2143,0.1303
5,2017-07-31,1,BREAD/BAKERY,370.9940,1,0.0000,153.8080,365.3500,339.9302,377.1632
6,2017-07-31,1,CELEBRATION,3.0000,1,0.0000,0.0000,22.0000,16.4286,13.7479
7,2017-07-31,1,CLEANING,678.0000,1,0.0000,238.0000,646.0000,663.7857,682.1722
8,2017-07-31,1,DAIRY,727.0000,1,0.0000,316.0000,758.0000,690.6429,667.0546
9,2017-07-31,1,DELI,189.1100,1,0.0000,54.2960,157.2490,132.5167,137.4665


Validation prediction rows: 28,512
Baselines created: 5


# 12.Baseline Metric Evaluation

This section evaluates baselines with technical and business-oriented metrics.


In [ ]:
def calculate_rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_true = np.maximum(y_true, 0)
    y_pred = np.maximum(y_pred, 0)

    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def calculate_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return np.mean(np.abs(y_pred - y_true))


def calculate_wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return np.sum(np.abs(y_pred - y_true)) / denominator


def calculate_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return np.mean(y_pred - y_true)


def calculate_total_bias_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(y_true)

    if denominator == 0:
        return np.nan

    return np.sum(y_pred - y_true) / denominator


baseline_descriptions = {
    "baseline_zero": "Predicts zero sales for every row.",
    "baseline_last_observed": "Repeats the last observed sales value before the validation horizon.",
    "baseline_last_7_day_pattern": "Repeats the last known weekly pattern before the validation horizon.",
    "baseline_rolling_mean_28_origin": "Uses the average sales of the last 28 days before the validation horizon.",
    "baseline_store_family_weekday_avg": "Uses the historical average by store, family and weekday.",
}

metrics_rows = []

y_true = baseline_predictions_valid[TARGET_COLUMN].values

for baseline_col in BASELINE_COLUMNS:
    y_pred = baseline_predictions_valid[baseline_col].values

    metrics_rows.append(
        {
            "baseline": baseline_col,
            "description": baseline_descriptions.get(baseline_col, ""),
            "rmsle": calculate_rmsle(y_true, y_pred),
            "mae": calculate_mae(y_true, y_pred),
            "wape": calculate_wape(y_true, y_pred),
            "bias": calculate_bias(y_true, y_pred),
            "total_bias_pct": calculate_total_bias_pct(y_true, y_pred),
            "actual_total_sales": np.sum(y_true),
            "predicted_total_sales": np.sum(y_pred),
        }
    )

baseline_metrics = pd.DataFrame(metrics_rows)

baseline_metrics["rmsle_rank"] = (
    baseline_metrics["rmsle"].rank(method="dense").astype(int)
)
baseline_metrics["mae_rank"] = baseline_metrics["mae"].rank(method="dense").astype(int)
baseline_metrics["wape_rank"] = (
    baseline_metrics["wape"].rank(method="dense").astype(int)
)

baseline_metrics = baseline_metrics.sort_values(["rmsle", "wape", "mae"]).reset_index(
    drop=True
)

baseline_metrics_display = baseline_metrics.copy()

for col in ["rmsle", "mae", "wape", "bias", "total_bias_pct"]:
    baseline_metrics_display[col] = baseline_metrics_display[col].round(4)

baseline_metrics_display["actual_total_sales"] = baseline_metrics_display[
    "actual_total_sales"
].round(2)
baseline_metrics_display["predicted_total_sales"] = baseline_metrics_display[
    "predicted_total_sales"
].round(2)

display(baseline_metrics_display)

best_baseline = baseline_metrics.iloc[0]["baseline"]
best_rmsle = baseline_metrics.iloc[0]["rmsle"]
best_wape = baseline_metrics.iloc[0]["wape"]

print(f"Best baseline by RMSLE: {best_baseline}")
print(f"Best RMSLE: {best_rmsle:.4f}")
print(f"Best WAPE: {best_wape:.4f}")


,baseline,description,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales,rmsle_rank,mae_rank,wape_rank
0,baseline_rolling_mean_28_origin,Uses the average sales of the last 28 days bef...,0.5216,97.2231,0.2081,6.2760,0.0134,"13,319,179.7800","13,498,121.0000",1,2,2
1,baseline_last_7_day_pattern,Repeats the last known weekly pattern before t...,0.6170,96.5354,0.2067,1.9146,0.0041,"13,319,179.7800","13,373,770.0000",2,1,1
2,baseline_last_observed,Repeats the last observed sales value before t...,0.6595,207.5058,0.4442,163.4701,0.3499,"13,319,179.7800","17,980,040.0000",3,4,4
3,baseline_store_family_weekday_avg,"Uses the historical average by store, family a...",0.6916,142.7926,0.3057,-113.5773,-0.2431,"13,319,179.7800","10,080,863.0000",4,3,3
4,baseline_zero,Predicts zero sales for every row.,4.4195,467.1429,1.0000,-467.1429,-1.0000,"13,319,179.7800",0.0000,5,5,5


Best baseline by RMSLE: baseline_rolling_mean_28_origin
Best RMSLE: 0.5216
Best WAPE: 0.2081


### Section conclusion

The baseline forecasts were evaluated using RMSLE, MAE, WAPE and bias.

RMSLE is used as the main technical metric because it is aligned with the Kaggle competition objective. MAE and WAPE provide a more interpretable business view of the forecast error, while bias indicates whether each baseline tends to overestimate or underestimate demand.

The best baseline by RMSLE becomes the minimum benchmark that future LightGBM and Prophet models must beat.


# 13.Baseline Error Analysis

This section reviews where the best baseline succeeds and where it creates operational risk.


In [ ]:
best_baseline = baseline_metrics.sort_values(["rmsle", "wape", "mae"]).iloc[0][
    "baseline"
]

error_analysis_df = baseline_predictions_valid[
    KEY_COLUMNS
    + [TARGET_COLUMN, "onpromotion", "day_of_week", "horizon_day", best_baseline]
].copy()

error_analysis_df = error_analysis_df.rename(columns={best_baseline: "prediction"})

error_analysis_df["absolute_error"] = (
    error_analysis_df["prediction"] - error_analysis_df[TARGET_COLUMN]
).abs()

error_analysis_df["error"] = (
    error_analysis_df["prediction"] - error_analysis_df[TARGET_COLUMN]
)

error_analysis_df["squared_log_error"] = (
    np.log1p(error_analysis_df["prediction"].clip(lower=0))
    - np.log1p(error_analysis_df[TARGET_COLUMN].clip(lower=0))
) ** 2

error_analysis_df["promotion_flag"] = np.where(
    error_analysis_df["onpromotion"] > 0,
    "with_promotion",
    "without_promotion",
)


def calculate_segment_metrics(df, group_columns):
    segment_rows = []

    for group_values, group_df in df.groupby(group_columns):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        y_true_group = group_df[TARGET_COLUMN].values
        y_pred_group = group_df["prediction"].values

        row = {
            "rows": len(group_df),
            "actual_total_sales": np.sum(y_true_group),
            "predicted_total_sales": np.sum(y_pred_group),
            "rmsle": calculate_rmsle(y_true_group, y_pred_group),
            "mae": calculate_mae(y_true_group, y_pred_group),
            "wape": calculate_wape(y_true_group, y_pred_group),
            "bias": calculate_bias(y_true_group, y_pred_group),
            "total_bias_pct": calculate_total_bias_pct(y_true_group, y_pred_group),
        }

        for col, value in zip(group_columns, group_values):
            row[col] = value

        segment_rows.append(row)

    segment_metrics = pd.DataFrame(segment_rows)

    ordered_columns = group_columns + [
        "rows",
        "actual_total_sales",
        "predicted_total_sales",
        "rmsle",
        "mae",
        "wape",
        "bias",
        "total_bias_pct",
    ]

    return segment_metrics[ordered_columns]


baseline_metrics_by_date = (
    calculate_segment_metrics(
        error_analysis_df,
        ["date"],
    )
    .sort_values("rmsle", ascending=False)
    .reset_index(drop=True)
)

baseline_metrics_by_store = (
    calculate_segment_metrics(
        error_analysis_df,
        ["store_nbr"],
    )
    .sort_values("rmsle", ascending=False)
    .reset_index(drop=True)
)

baseline_metrics_by_family = (
    calculate_segment_metrics(
        error_analysis_df,
        ["family"],
    )
    .sort_values("rmsle", ascending=False)
    .reset_index(drop=True)
)

baseline_metrics_by_promotion = (
    calculate_segment_metrics(
        error_analysis_df,
        ["promotion_flag"],
    )
    .sort_values("rmsle", ascending=False)
    .reset_index(drop=True)
)

baseline_metrics_by_horizon_day = (
    calculate_segment_metrics(
        error_analysis_df,
        ["horizon_day"],
    )
    .sort_values("horizon_day")
    .reset_index(drop=True)
)


display_cols = [
    "rows",
    "actual_total_sales",
    "predicted_total_sales",
    "rmsle",
    "mae",
    "wape",
    "bias",
    "total_bias_pct",
]


def display_rounded(df, n_rows=None):
    display_df = df.copy()
    numeric_cols = display_df.select_dtypes(
        include=["float", "float32", "float64"]
    ).columns
    display_df[numeric_cols] = display_df[numeric_cols].round(4)

    if n_rows is not None:
        display(display_df.head(n_rows))
    else:
        display(display_df)


print(f"Best baseline selected for error analysis: {best_baseline}")

print("\nWorst validation dates by RMSLE:")
display_rounded(baseline_metrics_by_date, n_rows=10)

print("\nWorst stores by RMSLE:")
display_rounded(baseline_metrics_by_store, n_rows=10)

print("\nWorst families by RMSLE:")
display_rounded(baseline_metrics_by_family, n_rows=10)

print("\nPerformance by promotion flag:")
display_rounded(baseline_metrics_by_promotion)

print("\nPerformance by forecast horizon day:")
display_rounded(baseline_metrics_by_horizon_day)


Best baseline selected for error analysis: baseline_rolling_mean_28_origin

Worst validation dates by RMSLE:


,date,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct
0,2017-08-14,1782,"760,922.4061","843,632.5000",0.5618,79.2970,0.1857,46.4142,0.1087
1,2017-08-11,1782,"826,373.7220","843,632.5000",0.5507,92.8045,0.2001,9.6851,0.0209
2,2017-08-13,1782,"865,639.6775","843,632.5000",0.5452,83.3699,0.1716,-12.3497,-0.0254
3,2017-08-08,1782,"717,766.3491","843,632.5000",0.5436,111.7260,0.2774,70.6320,0.1754
4,2017-08-06,1782,"1,049,559.1643","843,632.5000",0.5430,146.9607,0.2495,-115.5593,-0.1962
5,2017-08-10,1782,"651,386.9120","843,632.5000",0.5422,130.5002,0.3570,107.8820,0.2951
6,2017-08-12,1782,"792,630.5351","843,632.5000",0.5383,88.0447,0.1979,28.6207,0.0643
7,2017-08-15,1782,"762,661.9359","843,632.5000",0.5355,102.4771,0.2394,45.4381,0.1062
8,2017-08-07,1782,"797,464.9638","843,632.5000",0.5247,69.9585,0.1563,25.9077,0.0579
9,2017-08-09,1782,"734,139.6740","843,632.5000",0.5216,103.1230,0.2503,61.4438,0.1491



Worst stores by RMSLE:


,store_nbr,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct
0,50,528,"326,493.2400","346,601.5000",0.9147,116.4624,0.1883,38.0839,0.0616
1,47,528,"581,797.1710","606,709.2500",0.8815,215.3197,0.1954,47.1819,0.0428
2,48,528,"403,740.9601","402,595.8438",0.8574,189.5408,0.2479,-2.1687,-0.0028
3,44,528,"656,651.3530","723,267.6250",0.7904,268.9446,0.2163,126.1671,0.1014
4,20,528,"288,872.7331","250,589.5625",0.7144,137.2624,0.2509,-72.5060,-0.1325
5,9,528,"303,853.3809","276,694.9062",0.6902,114.3834,0.1988,-51.4364,-0.0894
6,19,528,"150,509.4550","146,131.1250",0.6332,48.1923,0.1691,-8.2923,-0.0291
7,18,528,"162,414.1110","166,268.7188",0.5865,73.1475,0.2378,7.3004,0.0237
8,14,528,"136,648.1410","129,410.2031",0.5729,65.6186,0.2535,-13.7082,-0.0530
9,26,528,"74,602.8770","80,547.8828",0.5709,47.9167,0.3391,11.2595,0.0797



Worst families by RMSLE:


,family,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct
0,SCHOOL AND OFFICE SUPPLIES,864,"51,795.0000","3,966.2856",1.6422,55.9101,0.9326,-55.3573,-0.9234
1,"LIQUOR,WINE,BEER",864,"84,091.0000","80,656.5703",0.6937,45.0312,0.4627,-3.9750,-0.0408
2,GROCERY II,864,"25,031.0000","29,641.7148",0.6795,12.6705,0.4373,5.3365,0.1842
3,LINGERIE,864,"6,716.0000","5,418.8574",0.6623,4.0846,0.5255,-1.5013,-0.1931
4,CELEBRATION,864,"10,768.0000","11,615.9990",0.6074,5.6066,0.4499,0.9815,0.0788
5,LADIESWEAR,864,"8,711.0000","9,832.0000",0.5802,4.5050,0.4468,1.2975,0.1287
6,HARDWARE,864,"1,252.0000","1,272.5714",0.5557,1.1133,0.7683,0.0238,0.0164
7,SEAFOOD,864,"17,260.9190","17,428.5723",0.5543,5.2606,0.2633,0.1940,0.0097
8,BEAUTY,864,"5,757.0000","4,672.5718",0.5531,2.8070,0.4213,-1.2551,-0.1884
9,MAGAZINES,864,"6,146.0000","5,551.4287",0.5441,2.9640,0.4167,-0.6882,-0.0967



Performance by promotion flag:


,promotion_flag,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct
0,with_promotion,12481,"12,435,885.5011","12,553,619.0000",0.5583,206.5786,0.2073,9.4330,0.0095
1,without_promotion,16031,"883,294.2805","944,502.1875",0.4911,12.0839,0.2193,3.8181,0.0693



Performance by forecast horizon day:


,horizon_day,rows,actual_total_sales,predicted_total_sales,rmsle,mae,wape,bias,total_bias_pct
0,1,1782,"885,856.8409","843,632.5000",0.4724,68.2274,0.1372,-23.6949,-0.0477
1,2,1782,"988,527.7632","843,632.5000",0.4671,106.2712,0.1916,-81.3104,-0.1466
2,3,1782,"964,712.0161","843,632.5000",0.4742,95.5763,0.1765,-67.9458,-0.1255
3,4,1782,"728,068.4851","843,632.5000",0.5054,101.0414,0.2473,64.8508,0.1587
4,5,1782,"827,775.6861","843,632.5000",0.5123,82.8403,0.1783,8.8984,0.0192
5,6,1782,"965,693.6505","843,632.5000",0.4943,93.3515,0.1723,-68.4967,-0.1264
6,7,1782,"1,049,559.1643","843,632.5000",0.5430,146.9607,0.2495,-115.5593,-0.1962
7,8,1782,"797,464.9638","843,632.5000",0.5247,69.9585,0.1563,25.9077,0.0579
8,9,1782,"717,766.3491","843,632.5000",0.5436,111.7260,0.2774,70.6320,0.1754
9,10,1782,"734,139.6740","843,632.5000",0.5216,103.1230,0.2503,61.4438,0.1491


### Section conclusion

The best baseline selected by RMSLE is `baseline_rolling_mean_28_origin`.

This baseline provides a stable and competitive reference, with a validation RMSLE of `0.5216` and WAPE of `20.81%`. However, the segment-level analysis shows clear limitations.

The largest error appears in `SCHOOL AND OFFICE SUPPLIES`, where the baseline strongly underestimates demand. This suggests that recent historical averages are not enough to capture sharp seasonal or campaign-driven demand changes.

The worst stores by RMSLE include stores `50`, `47`, `48` and `44`, indicating that some locations have demand patterns that are harder to explain with a single historical average.

Promotion periods also show higher RMSLE than non-promotion periods, confirming that promotional activity introduces demand variation that simple baselines cannot fully capture.

Finally, the forecast horizon analysis shows that the rolling 28-day baseline produces a stable daily total, but it does not adapt well to day-specific demand changes across the 16-day horizon.

The next modeling stage should improve this benchmark by learning interactions between calendar patterns, promotions, store metadata, product family behavior, holidays and recent sales history.


# 14.Save Feature and Baseline Artifacts

This section saves reusable feature datasets, validation predictions, metrics, and baseline metadata.


In [ ]:
output_dirs_to_check = {
    "FEATURES_DIR": FEATURES_DIR,
    "PREDICTIONS_DIR": PREDICTIONS_DIR,
    "REPORTS_TABLES_DIR": REPORTS_TABLES_DIR,
    "BASELINES_MODELS_DIR": BASELINES_MODELS_DIR,
}

missing_output_dirs = {
    name: path for name, path in output_dirs_to_check.items() if not path.exists()
}

if missing_output_dirs:
    raise FileNotFoundError(
        "Missing output directories. This notebook does not create folders:\n"
        + "\n".join([f"- {name}: {path}" for name, path in missing_output_dirs.items()])
    )


baseline_train_features_path = FEATURES_DIR / "baseline_train_features.parquet"
baseline_valid_features_path = FEATURES_DIR / "baseline_valid_features.parquet"
baseline_test_features_path = FEATURES_DIR / "baseline_test_features.parquet"
baseline_full_features_path = FEATURES_DIR / "baseline_full_features.parquet"

baseline_validation_predictions_path = (
    PREDICTIONS_DIR / "baseline_validation_predictions.parquet"
)

baseline_metrics_path = REPORTS_TABLES_DIR / "baseline_metrics.csv"
baseline_metrics_by_date_path = REPORTS_TABLES_DIR / "baseline_metrics_by_date.csv"
baseline_metrics_by_store_path = REPORTS_TABLES_DIR / "baseline_metrics_by_store.csv"
baseline_metrics_by_family_path = REPORTS_TABLES_DIR / "baseline_metrics_by_family.csv"
baseline_metrics_by_promotion_path = (
    REPORTS_TABLES_DIR / "baseline_metrics_by_promotion.csv"
)
baseline_metrics_by_horizon_day_path = (
    REPORTS_TABLES_DIR / "baseline_metrics_by_horizon_day.csv"
)

baseline_leaderboard_path = BASELINES_MODELS_DIR / "baseline_leaderboard.csv"
baseline_config_path = BASELINES_MODELS_DIR / "baseline_config.json"


train_model_fe.to_parquet(baseline_train_features_path, index=False)
valid_fe.to_parquet(baseline_valid_features_path, index=False)
test_fe.to_parquet(baseline_test_features_path, index=False)
train_full_fe.to_parquet(baseline_full_features_path, index=False)

baseline_predictions_valid.to_parquet(baseline_validation_predictions_path, index=False)

baseline_metrics.to_csv(baseline_metrics_path, index=False)
baseline_metrics_by_date.to_csv(baseline_metrics_by_date_path, index=False)
baseline_metrics_by_store.to_csv(baseline_metrics_by_store_path, index=False)
baseline_metrics_by_family.to_csv(baseline_metrics_by_family_path, index=False)
baseline_metrics_by_promotion.to_csv(baseline_metrics_by_promotion_path, index=False)
baseline_metrics_by_horizon_day.to_csv(
    baseline_metrics_by_horizon_day_path, index=False
)

baseline_metrics.to_csv(baseline_leaderboard_path, index=False)

baseline_config = {
    "notebook": "03_feature_engineering_baselines.ipynb",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "prediction_grain": KEY_COLUMNS,
    "target_column": TARGET_COLUMN,
    "forecast_horizon_days": FORECAST_HORIZON_DAYS,
    "train_start_date": str(train_model_base["date"].min().date()),
    "train_end_date": str(TRAIN_END_DATE.date()),
    "valid_start_date": str(VALID_START_DATE.date()),
    "valid_end_date": str(VALID_END_DATE.date()),
    "test_start_date": str(TEST_START_DATE.date()),
    "test_end_date": str(TEST_END_DATE.date()),
    "baseline_columns": BASELINE_COLUMNS,
    "best_baseline": best_baseline,
    "best_baseline_rmsle": float(
        baseline_metrics.loc[
            baseline_metrics["baseline"] == best_baseline, "rmsle"
        ].iloc[0]
    ),
    "best_baseline_wape": float(
        baseline_metrics.loc[
            baseline_metrics["baseline"] == best_baseline, "wape"
        ].iloc[0]
    ),
    "calendar_features": CALENDAR_FEATURES,
    "historical_sales_features": HISTORICAL_SALES_FEATURES,
    "leakage_rules": leakage_rules.to_dict(orient="records"),
    "notes": [
        "Transactions are excluded because they are not available for the future test period.",
        "Historical sales features are created using time-safe calendar-based lookups.",
        "Validation uses the last 16 days of the available training period.",
        "No folders are created by this notebook.",
    ],
}

with open(baseline_config_path, "w", encoding="utf-8") as file:
    json.dump(baseline_config, file, indent=4)


saved_outputs = pd.DataFrame(
    [
        {
            "output_type": "features",
            "path": baseline_train_features_path,
            "exists": baseline_train_features_path.exists(),
        },
        {
            "output_type": "features",
            "path": baseline_valid_features_path,
            "exists": baseline_valid_features_path.exists(),
        },
        {
            "output_type": "features",
            "path": baseline_test_features_path,
            "exists": baseline_test_features_path.exists(),
        },
        {
            "output_type": "features",
            "path": baseline_full_features_path,
            "exists": baseline_full_features_path.exists(),
        },
        {
            "output_type": "predictions",
            "path": baseline_validation_predictions_path,
            "exists": baseline_validation_predictions_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_path,
            "exists": baseline_metrics_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_by_date_path,
            "exists": baseline_metrics_by_date_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_by_store_path,
            "exists": baseline_metrics_by_store_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_by_family_path,
            "exists": baseline_metrics_by_family_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_by_promotion_path,
            "exists": baseline_metrics_by_promotion_path.exists(),
        },
        {
            "output_type": "metrics",
            "path": baseline_metrics_by_horizon_day_path,
            "exists": baseline_metrics_by_horizon_day_path.exists(),
        },
        {
            "output_type": "baseline_artifact",
            "path": baseline_leaderboard_path,
            "exists": baseline_leaderboard_path.exists(),
        },
        {
            "output_type": "baseline_artifact",
            "path": baseline_config_path,
            "exists": baseline_config_path.exists(),
        },
    ]
)

display(saved_outputs)

print("Baseline outputs saved successfully.")


,output_type,path,exists
0,features,....,True
1,features,....,True
2,features,....,True
3,features,....,True
4,predictions,....,True
5,metrics,....,True
6,metrics,....,True
7,metrics,....,True
8,metrics,....,True
9,metrics,....,True


Baseline outputs saved successfully.


# 15.Notebook Closure Criteria

This section verifies that the notebook produced the expected deliverables.


In [14]:
best_baseline_row = baseline_metrics.sort_values(["rmsle", "wape", "mae"]).iloc[0]

closing_summary = pd.DataFrame(
    [
        {
            "closing_criterion": "Prediction grain validated",
            "status": "completed",
            "evidence": "date + store_nbr + family is unique in train and test",
        },
        {
            "closing_criterion": "Temporal split created",
            "status": "completed",
            "evidence": f"Validation window: {VALID_START_DATE.date()} to {VALID_END_DATE.date()}",
        },
        {
            "closing_criterion": "Calendar features created",
            "status": "completed",
            "evidence": f"{len(CALENDAR_FEATURES)} calendar features created",
        },
        {
            "closing_criterion": "Historical sales features created",
            "status": "completed",
            "evidence": f"{len(HISTORICAL_SALES_FEATURES)} time-safe historical features created",
        },
        {
            "closing_criterion": "Leakage risks controlled",
            "status": "completed",
            "evidence": "Validation and test historical features use only available past sales",
        },
        {
            "closing_criterion": "Baselines evaluated",
            "status": "completed",
            "evidence": f"{len(BASELINE_COLUMNS)} baselines evaluated",
        },
        {
            "closing_criterion": "Best baseline selected",
            "status": "completed",
            "evidence": best_baseline,
        },
        {
            "closing_criterion": "Outputs saved",
            "status": "completed",
            "evidence": "Features, predictions, metrics and baseline artifacts saved",
        },
        {
            "closing_criterion": "Ready for advanced modeling",
            "status": "completed",
            "evidence": "LightGBM can use this dataset and benchmark as starting point",
        },
    ]
)

best_baseline_summary = pd.DataFrame(
    [
        {
            "best_baseline": best_baseline_row["baseline"],
            "rmsle": best_baseline_row["rmsle"],
            "mae": best_baseline_row["mae"],
            "wape": best_baseline_row["wape"],
            "bias": best_baseline_row["bias"],
            "total_bias_pct": best_baseline_row["total_bias_pct"],
            "actual_total_sales": best_baseline_row["actual_total_sales"],
            "predicted_total_sales": best_baseline_row["predicted_total_sales"],
        }
    ]
)

display(closing_summary)
display(best_baseline_summary.round(4))

print("Notebook 03 completed successfully.")
print(f"Minimum benchmark for future models: {best_baseline}")
print(f"Benchmark RMSLE: {best_baseline_row['rmsle']:.4f}")
print(f"Benchmark WAPE: {best_baseline_row['wape']:.4f}")


,closing_criterion,status,evidence
0,Prediction grain validated,completed,date + store_nbr + family is unique in train a...
1,Temporal split created,completed,Validation window: 2017-07-31 to 2017-08-15
2,Calendar features created,completed,8 calendar features created
3,Historical sales features created,completed,6 time-safe historical features created
4,Leakage risks controlled,completed,Validation and test historical features use on...
5,Baselines evaluated,completed,5 baselines evaluated
6,Best baseline selected,completed,baseline_rolling_mean_28_origin
7,Outputs saved,completed,"Features, predictions, metrics and baseline ar..."
8,Ready for advanced modeling,completed,LightGBM can use this dataset and benchmark as...


,best_baseline,rmsle,mae,wape,bias,total_bias_pct,actual_total_sales,predicted_total_sales
0,baseline_rolling_mean_28_origin,0.5216,97.2231,0.2081,6.2760,0.0134,"13,319,179.7816","13,498,121.0000"


Notebook 03 completed successfully.
Minimum benchmark for future models: baseline_rolling_mean_28_origin
Benchmark RMSLE: 0.5216
Benchmark WAPE: 0.2081


# 16.Final notebook conclusion

This notebook created the first reproducible modeling dataset for the Store Sales Forecasting project and established a clear baseline benchmark for future models.

The prediction grain was validated at `date + store_nbr + family`, and a strict 16-day temporal validation split was created to match the Kaggle forecasting horizon.

Basic calendar features and time-safe historical sales features were generated without using future sales information. `transactions` was excluded from predictive modeling because it is not available in the future test period.

Five simple baselines were evaluated. The best baseline by RMSLE was `baseline_rolling_mean_28_origin`, with a validation RMSLE of `0.5216` and WAPE of `20.81%`.

The error analysis showed that this baseline is stable but limited. It struggles with sharp demand changes, especially in `SCHOOL AND OFFICE SUPPLIES`, high-error stores, promotional periods and day-specific demand variation across the forecast horizon.

The next notebook should train a LightGBM model using the saved feature datasets and compare its performance against this baseline benchmark.
